# ColabFold Multimer Result Viewer

Reusable TREX1 binder review notebook for ColabFold multimer outputs. It scans result folders, computes aggregate binder quality metrics, ranks candidates, plots pLDDT-colored structures, and provides an interactive py3Dmol viewer.

The notebook only reads PDB/JSON/CSV/FASTA files. It intentionally does not load RFDiffusion `.trb` files because they are pickle-based.

In [1]:
from pathlib import Path
import csv
import json
import math
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import py3Dmol

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

# Path settings. These defaults assume Jupyter is launched from /export/data/sfan.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ColabFold").exists() and PROJECT_ROOT.name == "ColabFold":
    PROJECT_ROOT = PROJECT_ROOT.parent

COLABFOLD_ROOT = PROJECT_ROOT / "ColabFold/outputs/TREX1"
MPNN_FASTA = PROJECT_ROOT / "ProteinMPNN/outputs/TREX1/best_results/colabfold_multimer_best.fa"
RFDIFFUSION_ROOT = PROJECT_ROOT / "RFDiffusion/outputs/trex1_hotspot18_200"
MONOMER_ROOT = None

TARGET_CHAIN = "A"
BINDER_CHAIN = "B"
HOTSPOTS = [18, 200]
POCKET_RADIUS = 8.0
POCKET_CONTACT_CUTOFF = 8.0
POCKET_SURFACE_COVERAGE_PASS_CUTOFF = 0.35
POCKET_SPECIFICITY_PASS_CUTOFF = 0.45
INTERFACE_PAE_PASS_CUTOFF = 10.0
IPTM_PASS_CUTOFF = 0.80
BINDER_PLDDT_PASS_CUTOFF = 80.0

PASS_RULES = {
    "median_pae_max": 8.0,
    "median_iptm_min": 0.80,
    "binder_plddt_min": 80.0,
    "pose_rmsd_median_max": 5.0,
    "pose_rmsd_std_max": 2.0,
    "min_model_pass_fraction": 0.50,
}

FAIL_RULES = {
    "median_pae_min": 15.0,
    "median_iptm_max": 0.60,
    "binder_plddt_max": 70.0,
}

PLDDT_COLORS = {
    "very_high": "#1f3cff",
    "confident": "#18c6d9",
    "low": "#f0e442",
    "very_low": "#ff7d45",
}

CHAIN_COLORS = {
    TARGET_CHAIN: "#35d04c",
    BINDER_CHAIN: "#22c7cf",
}


AA3_TO_1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
}


## Parsing and Metrics

In [2]:
def plddt_color(value):
    if value >= 90:
        return PLDDT_COLORS["very_high"]
    if value >= 70:
        return PLDDT_COLORS["confident"]
    if value >= 50:
        return PLDDT_COLORS["low"]
    return PLDDT_COLORS["very_low"]


def parse_rank(path):
    match = re.search(r"rank_(\d+)", path.name)
    return int(match.group(1)) if match else 10**9


def parse_fasta_records(path):
    records = []
    if not path or not Path(path).exists():
        return records
    header = None
    seq_parts = []
    with Path(path).open() as handle:
        for raw in handle:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq_parts)))
                header = line[1:]
                seq_parts = []
            else:
                seq_parts.append(line)
    if header is not None:
        records.append((header, "".join(seq_parts)))
    return records


def parse_mpnn_header(header):
    # Example: run_noise0_45|T=0.1|sample=7|score=0.9095
    binder_id = header.split("|")[0].strip()
    score = math.nan
    match = re.search(r"(?:^|[|, ])score=([0-9.]+)", header)
    if match:
        score = float(match.group(1))
    return binder_id, score


def build_mpnn_sequence_map(path=MPNN_FASTA):
    seq_map = {}
    for header, sequence in parse_fasta_records(path):
        binder_id, score = parse_mpnn_header(header)
        binder_sequence = sequence.split(":")[-1]
        seq_map[binder_sequence] = {
            "binder_id": binder_id,
            "mpnn_score": score,
            "mpnn_header": header,
        }
    return seq_map


def read_colabfold_sequence(folder):
    csv_files = sorted(Path(folder).glob("*.csv"))
    if not csv_files:
        return None
    with csv_files[0].open(newline="") as handle:
        reader = csv.DictReader(handle)
        row = next(reader, None)
    if not row:
        return None
    return row.get("sequence")


def parse_pdb_atoms(path):
    atoms = defaultdict(lambda: defaultdict(list))
    ca = defaultdict(dict)
    residues_order = []
    seen_residues = set()
    residue_plddt = {}

    with Path(path).open() as handle:
        for line in handle:
            if not line.startswith(("ATOM  ", "HETATM")):
                continue
            atom_name = line[12:16].strip()
            element = line[76:78].strip() if len(line) >= 78 else ""
            if atom_name.startswith("H") or element == "H":
                continue
            chain = line[21].strip() or "_"
            try:
                residue = int(line[22:26])
                insertion = line[26].strip()
                xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])], dtype=float)
                bfactor = float(line[60:66])
            except ValueError as exc:
                raise ValueError(f"Could not parse ATOM line in {path}: {line.rstrip()}") from exc

            residue_key = (chain, residue, insertion)
            if residue_key not in seen_residues:
                residues_order.append(residue_key)
                seen_residues.add(residue_key)
            atoms[chain][residue].append((atom_name, xyz, bfactor))
            if atom_name == "CA":
                ca[chain][residue] = xyz
                residue_plddt[residue_key] = bfactor

    return atoms, ca, residues_order, residue_plddt


def chain_offsets_from_pdb(pdb_path):
    _atoms, _ca, residues_order, _residue_plddt = parse_pdb_atoms(pdb_path)
    offsets = {}
    for idx, (chain, residue, insertion) in enumerate(residues_order):
        offsets.setdefault(chain, []).append(idx)
    return offsets


def pairwise_distances(coords_a, coords_b):
    if len(coords_a) == 0 or len(coords_b) == 0:
        return np.empty((len(coords_a), len(coords_b)), dtype=float)
    return np.linalg.norm(coords_a[:, None, :] - coords_b[None, :, :], axis=2)


def stack_chain_heavy_atoms(atoms, chain):
    coords = []
    residue_ids = []
    for residue, records in atoms.get(chain, {}).items():
        for _atom_name, xyz, _bfactor in records:
            coords.append(xyz)
            residue_ids.append(residue)
    if not coords:
        return np.empty((0, 3), dtype=float), np.array([], dtype=int)
    return np.vstack(coords), np.array(residue_ids, dtype=int)


def normalize_hotspot_definitions(hotspots=HOTSPOTS):
    normalized = []
    for hotspot in hotspots:
        if isinstance(hotspot, tuple):
            chain, residue = hotspot
        else:
            chain, residue = TARGET_CHAIN, hotspot
        normalized.append((str(chain), int(residue), f"hotspot{int(residue)}"))
    return normalized


def residue_min_distance_to_coords(residue_records, coords):
    if not residue_records or len(coords) == 0:
        return math.inf
    residue_coords = np.vstack([xyz for _atom_name, xyz, _bfactor in residue_records])
    return float(np.min(pairwise_distances(residue_coords, coords)))


def pdb_chain_residue_order_and_sequence(path, chain=TARGET_CHAIN):
    residues = []
    seen = set()
    sequence = []
    with Path(path).open() as handle:
        for line in handle:
            if not line.startswith("ATOM"):
                continue
            pdb_chain = line[21].strip() or "_"
            if pdb_chain != chain:
                continue
            try:
                residue = int(line[22:26])
            except ValueError:
                continue
            insertion = line[26].strip()
            key = (pdb_chain, residue, insertion)
            if key in seen:
                continue
            seen.add(key)
            residues.append((pdb_chain, residue, insertion))
            sequence.append(AA3_TO_1.get(line[17:20].strip(), "X"))
    return residues, "".join(sequence)


def map_reference_hotspots_to_pdb(pdb_path, reference_pdb, hotspots=HOTSPOTS):
    hotspot_defs = normalize_hotspot_definitions(hotspots)
    if not reference_pdb or not Path(reference_pdb).exists():
        mapped = [(chain, residue, label, chain, residue, False) for chain, residue, label in hotspot_defs]
        return mapped, ["No reference PDB available; treating hotspot numbers as ColabFold PDB numbering."]

    warnings = []
    mapped = []
    for ref_chain, ref_residue, label in hotspot_defs:
        ref_order, ref_seq = pdb_chain_residue_order_and_sequence(reference_pdb, ref_chain)
        pdb_order, pdb_seq = pdb_chain_residue_order_and_sequence(pdb_path, ref_chain)
        if not ref_order or not pdb_order:
            mapped.append((ref_chain, ref_residue, label, ref_chain, ref_residue, False))
            warnings.append(f"Could not map {ref_chain}{ref_residue}: chain missing in reference or prediction; using input numbering.")
            continue
        if len(ref_order) != len(pdb_order) or ref_seq != pdb_seq:
            mapped.append((ref_chain, ref_residue, label, ref_chain, ref_residue, False))
            warnings.append(f"Could not map {ref_chain}{ref_residue}: target sequences differ; using input numbering.")
            continue
        ref_lookup = {(chain, residue): idx for idx, (chain, residue, _insertion) in enumerate(ref_order)}
        idx = ref_lookup.get((ref_chain, ref_residue))
        if idx is None:
            mapped.append((ref_chain, ref_residue, label, ref_chain, ref_residue, False))
            warnings.append(f"Could not map {ref_chain}{ref_residue}: residue not found in reference; using input numbering.")
            continue
        pdb_chain, pdb_residue, _pdb_insertion = pdb_order[idx]
        mapped.append((pdb_chain, pdb_residue, label, ref_chain, ref_residue, True))
    return mapped, warnings


def format_hotspot_mapping(mapped_hotspots):
    parts = []
    for pdb_chain, pdb_residue, _label, ref_chain, ref_residue, was_mapped in mapped_hotspots:
        if was_mapped and (pdb_chain != ref_chain or pdb_residue != ref_residue):
            parts.append(f"{ref_chain}{ref_residue}->{pdb_chain}{pdb_residue}")
        else:
            parts.append(f"{ref_chain}{ref_residue}")
    return ", ".join(parts)


def compute_pocket_surface_retention(pdb_path, hotspots=HOTSPOTS, reference_pdb=None):
    atoms, _ca, _order, _plddt = parse_pdb_atoms(pdb_path)
    binder_coords, _binder_residue_ids = stack_chain_heavy_atoms(atoms, BINDER_CHAIN)
    target_atoms = atoms.get(TARGET_CHAIN, {})
    mapped_hotspots, mapping_warnings = map_reference_hotspots_to_pdb(pdb_path, reference_pdb, hotspots)

    hotspot_coords_by_label = {}
    for pdb_chain, pdb_residue, label, _ref_chain, _ref_residue, _was_mapped in mapped_hotspots:
        records = atoms.get(pdb_chain, {}).get(pdb_residue, [])
        hotspot_coords_by_label[label] = np.vstack([xyz for _atom_name, xyz, _bfactor in records]) if records else np.empty((0, 3))

    combined_hotspot_coords = [coords for coords in hotspot_coords_by_label.values() if len(coords)]
    combined_hotspot_coords = np.vstack(combined_hotspot_coords) if combined_hotspot_coords else np.empty((0, 3))

    pocket_residues = set()
    lobe_residues_by_label = {label: set() for _pdb_chain, _pdb_residue, label, _ref_chain, _ref_residue, _was_mapped in mapped_hotspots}
    for residue, records in target_atoms.items():
        if residue_min_distance_to_coords(records, combined_hotspot_coords) <= POCKET_RADIUS:
            pocket_residues.add(int(residue))
        for _pdb_chain, _pdb_residue, label, _ref_chain, _ref_residue, _was_mapped in mapped_hotspots:
            if residue_min_distance_to_coords(records, hotspot_coords_by_label[label]) <= POCKET_RADIUS:
                lobe_residues_by_label[label].add(int(residue))

    contacted_target_residues = set()
    for residue, records in target_atoms.items():
        if residue_min_distance_to_coords(records, binder_coords) <= POCKET_CONTACT_CUTOFF:
            contacted_target_residues.add(int(residue))

    contacted_pocket_residues = pocket_residues & contacted_target_residues
    pocket_total = len(pocket_residues)
    pocket_contacted = len(contacted_pocket_residues)
    pocket_surface_coverage = pocket_contacted / pocket_total if pocket_total else 0.0
    pocket_specificity = pocket_contacted / len(contacted_target_residues) if contacted_target_residues else 0.0

    lobe_coverages = []
    lobe_counts = {}
    for _pdb_chain, _pdb_residue, label, _ref_chain, _ref_residue, _was_mapped in mapped_hotspots:
        lobe_residues = lobe_residues_by_label[label]
        lobe_contacted = len(lobe_residues & contacted_target_residues)
        lobe_total = len(lobe_residues)
        lobe_coverage = lobe_contacted / lobe_total if lobe_total else 0.0
        lobe_coverages.append(lobe_coverage)
        lobe_counts[label] = {
            "contacted": int(lobe_contacted),
            "total": int(lobe_total),
            "coverage": float(lobe_coverage),
        }

    return {
        "pocket_contacted_residues": int(pocket_contacted),
        "pocket_total_residues": int(pocket_total),
        "pocket_surface_coverage": float(pocket_surface_coverage),
        "pocket_specificity": float(pocket_specificity),
        "pocket_min_lobe_coverage": float(min(lobe_coverages)) if lobe_coverages else 0.0,
        "all_contacted_target_residues": int(len(contacted_target_residues)),
        "contacted_pocket_residues": sorted(contacted_pocket_residues),
        "pocket_residues": sorted(pocket_residues),
        "lobe_counts": lobe_counts,
        "mapped_hotspots": mapped_hotspots,
        "hotspot_mapping": format_hotspot_mapping(mapped_hotspots),
        "hotspot_mapping_warnings": mapping_warnings,
    }


def median_int(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    return int(round(float(values.median()))) if len(values) else 0


def median_float(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    return float(values.median()) if len(values) else math.nan


def format_percent(value):
    return "NA" if not np.isfinite(value) else f"{100 * value:.0f}%"


def format_pocket_surface_summary(model_df, passed_models=None, total_models=None):
    if model_df.empty:
        return "surface 0/0 (NA); spec NA; model_pass 0/0"

    pocket_contacted = median_int(model_df["pocket_contacted_residues"])
    pocket_total = median_int(model_df["pocket_total_residues"])
    pocket_coverage = median_float(model_df["pocket_surface_coverage"])
    specificity = median_float(model_df["pocket_specificity"])
    if passed_models is None:
        passed_models = int(model_df["model_passed"].sum()) if "model_passed" in model_df else 0
    if total_models is None:
        total_models = len(model_df)
    return f"surface {pocket_contacted}/{pocket_total} ({format_percent(pocket_coverage)}); spec {format_percent(specificity)}; model_pass {passed_models}/{total_models}"


def kabsch_transform(mobile_points, reference_points):
    mobile_points = np.asarray(mobile_points, dtype=float)
    reference_points = np.asarray(reference_points, dtype=float)
    mobile_center = mobile_points.mean(axis=0)
    reference_center = reference_points.mean(axis=0)
    mobile_centered = mobile_points - mobile_center
    reference_centered = reference_points - reference_center
    covariance = mobile_centered.T @ reference_centered
    u, _s, vt = np.linalg.svd(covariance)
    rotation = u @ vt
    if np.linalg.det(rotation) < 0:
        vt[-1, :] *= -1
        rotation = u @ vt
    return mobile_center, reference_center, rotation


def apply_transform(coords, mobile_center, reference_center, rotation):
    return (np.asarray(coords, dtype=float) - mobile_center) @ rotation + reference_center


def pose_rmsd_to_reference(predicted_pdb, reference_pdb):
    if not reference_pdb or not Path(reference_pdb).exists():
        return math.nan
    _pred_atoms, pred_ca, _pred_order, _pred_plddt = parse_pdb_atoms(predicted_pdb)
    _ref_atoms, ref_ca, _ref_order, _ref_plddt = parse_pdb_atoms(reference_pdb)

    common_target = sorted(set(pred_ca.get(TARGET_CHAIN, {})) & set(ref_ca.get(TARGET_CHAIN, {})))
    common_binder = sorted(set(pred_ca.get(BINDER_CHAIN, {})) & set(ref_ca.get(BINDER_CHAIN, {})))
    if len(common_target) < 3 or len(common_binder) < 3:
        return math.nan

    mobile_target = np.vstack([pred_ca[TARGET_CHAIN][res] for res in common_target])
    reference_target = np.vstack([ref_ca[TARGET_CHAIN][res] for res in common_target])
    mobile_center, reference_center, rotation = kabsch_transform(mobile_target, reference_target)

    mobile_binder = np.vstack([pred_ca[BINDER_CHAIN][res] for res in common_binder])
    reference_binder = np.vstack([ref_ca[BINDER_CHAIN][res] for res in common_binder])
    aligned_binder = apply_transform(mobile_binder, mobile_center, reference_center, rotation)
    return float(np.sqrt(np.mean(np.sum((aligned_binder - reference_binder) ** 2, axis=1))))


def interface_pae_from_scores(score_json, pdb_path):
    pae = np.asarray(score_json.get("pae", []), dtype=float)
    if pae.ndim != 2 or pae.shape[0] != pae.shape[1]:
        return math.nan
    offsets = chain_offsets_from_pdb(pdb_path)
    target_idx = offsets.get(TARGET_CHAIN, [])
    binder_idx = offsets.get(BINDER_CHAIN, [])
    if not target_idx or not binder_idx:
        return math.nan
    cross_ab = pae[np.ix_(target_idx, binder_idx)].ravel()
    cross_ba = pae[np.ix_(binder_idx, target_idx)].ravel()
    cross = np.concatenate([cross_ab, cross_ba])
    return float(np.mean(cross)) if len(cross) else math.nan


def binder_plddt_median_from_scores(score_json, pdb_path):
    plddt = np.asarray(score_json.get("plddt", []), dtype=float)
    offsets = chain_offsets_from_pdb(pdb_path)
    binder_idx = offsets.get(BINDER_CHAIN, [])
    if len(plddt) and binder_idx and max(binder_idx) < len(plddt):
        return float(np.median(plddt[binder_idx]))

    _atoms, _ca, residues_order, residue_plddt = parse_pdb_atoms(pdb_path)
    binder_values = [residue_plddt[key] for key in residues_order if key[0] == BINDER_CHAIN and key in residue_plddt]
    return float(np.median(binder_values)) if binder_values else math.nan


def load_model_pairs(folder):
    folder = Path(folder)
    pdbs = {parse_rank(p): p for p in folder.glob("*_unrelaxed_rank_*.pdb")}
    scores = {parse_rank(p): p for p in folder.glob("*_scores_rank_*.json")}
    pairs = []
    for rank in sorted(set(pdbs) & set(scores)):
        pairs.append({"rank": rank, "pdb_path": pdbs[rank], "score_path": scores[rank]})
    return pairs


def find_reference_pdb(binder_id):
    if not binder_id or not RFDIFFUSION_ROOT.exists():
        return None
    candidate = RFDIFFUSION_ROOT / f"{binder_id}.pdb"
    return candidate if candidate.exists() else None


def find_monomer_plddt(binder_id):
    if MONOMER_ROOT is None:
        return pd.NA
    root = Path(MONOMER_ROOT)
    if not root.exists() or not binder_id:
        return pd.NA
    score_files = sorted(root.glob(f"**/*{binder_id}*scores*.json"))
    for score_file in score_files:
        try:
            data = json.loads(score_file.read_text())
            plddt = data.get("plddt")
            if plddt:
                return float(np.median(np.asarray(plddt, dtype=float)))
        except Exception:
            continue
    return pd.NA


In [3]:
def analyze_result_folder(folder, mpnn_seq_map=None):
    folder = Path(folder)
    if mpnn_seq_map is None:
        mpnn_seq_map = build_mpnn_sequence_map()

    sequence = read_colabfold_sequence(folder)
    binder_sequence = sequence.split(":")[-1] if sequence and ":" in sequence else None
    mpnn_match = mpnn_seq_map.get(binder_sequence, {}) if binder_sequence else {}
    binder_id = mpnn_match.get("binder_id", folder.name)
    mpnn_score = mpnn_match.get("mpnn_score", math.nan)
    reference_pdb = find_reference_pdb(binder_id)
    model_pairs = load_model_pairs(folder)

    model_records = []
    for pair in model_pairs:
        with pair["score_path"].open() as handle:
            score_json = json.load(handle)
        interface_pae = interface_pae_from_scores(score_json, pair["pdb_path"])
        binder_plddt = binder_plddt_median_from_scores(score_json, pair["pdb_path"])
        pocket_metrics = compute_pocket_surface_retention(pair["pdb_path"], reference_pdb=reference_pdb)
        pose_rmsd = pose_rmsd_to_reference(pair["pdb_path"], reference_pdb)
        iptm = float(score_json.get("iptm", math.nan))
        pocket_passed = (
            pocket_metrics["pocket_surface_coverage"] >= POCKET_SURFACE_COVERAGE_PASS_CUTOFF
            and pocket_metrics["pocket_specificity"] >= POCKET_SPECIFICITY_PASS_CUTOFF
        )
        model_passed = (
            np.isfinite(iptm) and iptm >= IPTM_PASS_CUTOFF
            and np.isfinite(interface_pae) and interface_pae <= INTERFACE_PAE_PASS_CUTOFF
            and np.isfinite(binder_plddt) and binder_plddt >= BINDER_PLDDT_PASS_CUTOFF
            and pocket_passed
        )
        record = {
            "folder": folder.name,
            "binder_id": binder_id,
            "rank": pair["rank"],
            "pdb_path": str(pair["pdb_path"]),
            "score_path": str(pair["score_path"]),
            "ipTM": iptm,
            "pAE_interaction": interface_pae,
            "binder_pLDDT_median": binder_plddt,
            "pose_RMSD": pose_rmsd,
            "pocket_contacted_residues": pocket_metrics["pocket_contacted_residues"],
            "pocket_total_residues": pocket_metrics["pocket_total_residues"],
            "pocket_surface_coverage": pocket_metrics["pocket_surface_coverage"],
            "pocket_min_lobe_coverage": pocket_metrics["pocket_min_lobe_coverage"],
            "pocket_specificity": pocket_metrics["pocket_specificity"],
            "all_contacted_target_residues": pocket_metrics["all_contacted_target_residues"],
            "hotspot_mapping": pocket_metrics["hotspot_mapping"],
            "hotspot_mapping_warnings": "; ".join(pocket_metrics["hotspot_mapping_warnings"]),
            "model_passed": bool(model_passed),
        }
        for _chain, _residue, label in normalize_hotspot_definitions(HOTSPOTS):
            lobe = pocket_metrics["lobe_counts"].get(label, {"contacted": 0, "total": 0, "coverage": 0.0})
            record[f"{label}_lobe_contacted"] = lobe["contacted"]
            record[f"{label}_lobe_total"] = lobe["total"]
            record[f"{label}_lobe_coverage"] = lobe["coverage"]
        model_records.append(record)

    model_df = pd.DataFrame.from_records(model_records)
    total_models = len(model_df)
    if total_models == 0:
        return {
            "folder": folder.name,
            "folder_path": str(folder),
            "binder_id": binder_id,
            "mpnn_score": mpnn_score,
            "best_ipTM": math.nan,
            "median_ipTM": math.nan,
            "best_pAE_interaction": math.nan,
            "median_pAE_interaction": math.nan,
            "binder_pLDDT_median": math.nan,
            "pose_RMSD_median": math.nan,
            "pose_RMSD_std": math.nan,
            "hotspot_contacts_retained": "surface 0/0 (NA); lobes NA; spec NA",
            "pocket_surface_coverage_median": math.nan,
            "pocket_min_lobe_coverage_median": math.nan,
            "pocket_specificity_median": math.nan,
            "num_models_passed / total_models": "0/0",
            "monomer_pLDDT": pd.NA,
            "final_decision": "FAIL",
            "reference_pdb": str(reference_pdb) if reference_pdb else None,
            "model_df": model_df,
        }

    passed_models = int(model_df["model_passed"].sum())
    monomer_plddt = find_monomer_plddt(binder_id)

    summary = {
        "folder": folder.name,
        "folder_path": str(folder),
        "binder_id": binder_id,
        "mpnn_score": float(mpnn_score) if np.isfinite(mpnn_score) else math.nan,
        "best_ipTM": float(model_df["ipTM"].max()),
        "median_ipTM": float(model_df["ipTM"].median()),
        "best_pAE_interaction": float(model_df["pAE_interaction"].min()),
        "median_pAE_interaction": float(model_df["pAE_interaction"].median()),
        "binder_pLDDT_median": float(model_df["binder_pLDDT_median"].median()),
        "pose_RMSD_median": float(model_df["pose_RMSD"].median()) if model_df["pose_RMSD"].notna().any() else math.nan,
        "pose_RMSD_std": float(model_df["pose_RMSD"].std(ddof=0)) if model_df["pose_RMSD"].notna().any() else math.nan,
        "hotspot_contacts_retained": format_pocket_surface_summary(model_df, passed_models=passed_models, total_models=total_models),
        "pocket_surface_coverage_median": median_float(model_df["pocket_surface_coverage"]),
        "pocket_min_lobe_coverage_median": median_float(model_df["pocket_min_lobe_coverage"]),
        "pocket_specificity_median": median_float(model_df["pocket_specificity"]),
        "num_models_passed / total_models": f"{passed_models}/{total_models}",
        "monomer_pLDDT": monomer_plddt,
        "reference_pdb": str(reference_pdb) if reference_pdb else None,
        "model_df": model_df,
    }
    summary["final_decision"] = decide_final(summary, passed_models, total_models)
    return summary


def decide_final(summary, passed_models, total_models):
    if total_models == 0:
        return "FAIL"
    pass_fraction = passed_models / total_models
    median_pae = summary["median_pAE_interaction"]
    median_iptm = summary["median_ipTM"]
    binder_plddt = summary["binder_pLDDT_median"]
    pose_rmsd_median = summary["pose_RMSD_median"]
    pose_rmsd_std = summary["pose_RMSD_std"]
    pocket_coverage = summary["pocket_surface_coverage_median"]
    pocket_specificity = summary["pocket_specificity_median"]

    if (
        (np.isfinite(median_pae) and median_pae > FAIL_RULES["median_pae_min"])
        or (np.isfinite(median_iptm) and median_iptm < FAIL_RULES["median_iptm_max"])
        or (np.isfinite(binder_plddt) and binder_plddt < FAIL_RULES["binder_plddt_max"])
        or passed_models == 0
        or (np.isfinite(pocket_coverage) and pocket_coverage < 0.25)
        or (np.isfinite(pocket_specificity) and pocket_specificity < 0.35)
    ):
        return "FAIL"

    if (
        np.isfinite(median_pae) and median_pae <= PASS_RULES["median_pae_max"]
        and np.isfinite(median_iptm) and median_iptm >= PASS_RULES["median_iptm_min"]
        and np.isfinite(binder_plddt) and binder_plddt >= PASS_RULES["binder_plddt_min"]
        and np.isfinite(pose_rmsd_median) and pose_rmsd_median <= PASS_RULES["pose_rmsd_median_max"]
        and np.isfinite(pose_rmsd_std) and pose_rmsd_std <= PASS_RULES["pose_rmsd_std_max"]
        and np.isfinite(pocket_coverage) and pocket_coverage >= POCKET_SURFACE_COVERAGE_PASS_CUTOFF
        and np.isfinite(pocket_specificity) and pocket_specificity >= POCKET_SPECIFICITY_PASS_CUTOFF
        and pass_fraction >= PASS_RULES["min_model_pass_fraction"]
    ):
        return "PASS"
    return "REVIEW"


def scan_result_folders(root=COLABFOLD_ROOT):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"ColabFold root does not exist: {root}")
    folders = [p for p in sorted(root.iterdir()) if p.is_dir() and load_model_pairs(p)]
    if not folders:
        raise FileNotFoundError(f"No ColabFold result folders with PDB/score pairs found in {root}")
    return folders


def analyze_all_results():
    mpnn_seq_map = build_mpnn_sequence_map(MPNN_FASTA)
    summaries = [analyze_result_folder(folder, mpnn_seq_map=mpnn_seq_map) for folder in scan_result_folders()]
    display_records = []
    model_tables = {}
    for summary in summaries:
        model_tables[summary["folder_path"]] = summary.pop("model_df")
        display_records.append(summary)
    df = pd.DataFrame.from_records(display_records)
    df["monomer_sort"] = pd.to_numeric(df["monomer_pLDDT"], errors="coerce")
    df = df.sort_values(
        [
            "median_pAE_interaction",
            "median_ipTM",
            "binder_pLDDT_median",
            "pose_RMSD_median",
            "pose_RMSD_std",
            "monomer_sort",
        ],
        ascending=[True, False, False, True, True, False],
        na_position="last",
    ).drop(columns=["monomer_sort"]).reset_index(drop=True)
    return df, model_tables


def format_summary_table(df):
    cols = [
        "binder_id",
        "mpnn_score",
        "best_ipTM",
        "median_ipTM",
        "best_pAE_interaction",
        "median_pAE_interaction",
        "binder_pLDDT_median",
        "pose_RMSD_median",
        "pose_RMSD_std",
        "hotspot_contacts_retained",
        "num_models_passed / total_models",
        "monomer_pLDDT",
        "final_decision",
        "folder",
    ]
    return df[cols].style.format({
        "mpnn_score": "{:.4f}",
        "best_ipTM": "{:.3f}",
        "median_ipTM": "{:.3f}",
        "best_pAE_interaction": "{:.2f}",
        "median_pAE_interaction": "{:.2f}",
        "binder_pLDDT_median": "{:.1f}",
        "pose_RMSD_median": "{:.2f}",
        "pose_RMSD_std": "{:.2f}",
    })


## Ranked Summary Table

`hotspot_contacts_retained` reports whether the binder fills the A18/A200-defined pocket surface: median pocket-surface coverage, pocket-specificity, and how many ranked models pass the pocket-fill rule. A18/A200 lobe coverage remains in the per-model audit table for diagnosis, but it is no longer a pass/fail requirement.

In [4]:
all_results_df, model_tables = analyze_all_results()
format_summary_table(all_results_df)

,binder_id,mpnn_score,best_ipTM,median_ipTM,best_pAE_interaction,median_pAE_interaction,binder_pLDDT_median,pose_RMSD_median,pose_RMSD_std,hotspot_contacts_retained,num_models_passed / total_models,monomer_pLDDT,final_decision,folder
0,run_noise0_45,0.9095,0.870,0.860,5.65,5.98,96.9,13.64,0.82,1/2,0/20,,FAIL,TREX1_top_01_9c0cd


## Per-Model Audit Table

Use the folder dropdown to inspect every model/seed contributing to a candidate summary.

In [5]:
folder_options = [(f"{row.binder_id} | {row.folder}", row.folder_path) for row in all_results_df.itertuples()]
folder_dropdown = widgets.Dropdown(options=folder_options, description="Result:", layout=widgets.Layout(width="650px"))
model_audit_output = widgets.Output()


def show_model_audit(change=None):
    with model_audit_output:
        clear_output(wait=True)
        model_df = model_tables.get(folder_dropdown.value, pd.DataFrame()).copy()
        if model_df.empty:
            print("No model records found.")
            return
        display_cols = [
            "rank",
            "ipTM",
            "pAE_interaction",
            "binder_pLDDT_median",
            "pose_RMSD",
            "pocket_contacted_residues",
            "pocket_total_residues",
            "pocket_surface_coverage",
            "hotspot18_lobe_coverage",
            "hotspot200_lobe_coverage",
            "pocket_specificity",
            "hotspot_mapping",
            "model_passed",
            "pdb_path",
        ]
        display(
            model_df[display_cols]
            .sort_values("rank")
            .style.format({
                "ipTM": "{:.3f}",
                "pAE_interaction": "{:.2f}",
                "binder_pLDDT_median": "{:.1f}",
                "pose_RMSD": "{:.2f}",
                "pocket_surface_coverage": "{:.2%}",
                "hotspot18_lobe_coverage": "{:.2%}",
                "hotspot200_lobe_coverage": "{:.2%}",
                "pocket_specificity": "{:.2%}",
            })
        )

folder_dropdown.observe(show_model_audit, names="value")
show_model_audit()
display(widgets.VBox([folder_dropdown, model_audit_output]))


## 2D Structure Overview

In [6]:
def ca_trace_by_chain(pdb_path):
    _atoms, _ca, residues_order, residue_plddt = parse_pdb_atoms(pdb_path)
    traces = defaultdict(list)
    for key in residues_order:
        chain, residue, insertion = key
        if key in residue_plddt and residue in _ca.get(chain, {}):
            traces[chain].append({
                "residue": residue,
                "xyz": _ca[chain][residue],
                "plddt": residue_plddt[key],
            })
    return traces


def set_equal_3d_axes(ax, coords):
    coords = np.asarray(coords)
    if coords.size == 0:
        return
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)
    center = (mins + maxs) / 2
    radius = max(maxs - mins) / 2
    if radius <= 0:
        radius = 1.0
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)


def style_structure_axis(ax, title):
    ax.set_title(title, fontsize=11)
    ax.set_axis_off()
    ax.view_init(elev=18, azim=-70)


def plot_chain_colored(ax, pdb_path, title="colored by chain"):
    traces = ca_trace_by_chain(pdb_path)
    all_coords = []
    for chain, records in traces.items():
        coords = np.vstack([r["xyz"] for r in records]) if records else np.empty((0, 3))
        if len(coords) == 0:
            continue
        all_coords.extend(coords)
        ax.plot(
            coords[:, 0], coords[:, 1], coords[:, 2],
            color=CHAIN_COLORS.get(chain, "#888888"),
            linewidth=3.0,
            solid_capstyle="round",
        )
    set_equal_3d_axes(ax, all_coords)
    style_structure_axis(ax, title)


def plot_plddt_colored(ax, pdb_path, title="colored by pLDDT"):
    traces = ca_trace_by_chain(pdb_path)
    all_coords = []
    for chain, records in traces.items():
        if len(records) < 2:
            continue
        coords = np.vstack([r["xyz"] for r in records])
        plddts = np.asarray([r["plddt"] for r in records])
        all_coords.extend(coords)
        segments = np.stack([coords[:-1], coords[1:]], axis=1)
        segment_colors = [plddt_color(float((a + b) / 2)) for a, b in zip(plddts[:-1], plddts[1:])]
        collection = Line3DCollection(segments, colors=segment_colors, linewidths=3.0)
        ax.add_collection3d(collection)
    set_equal_3d_axes(ax, all_coords)
    style_structure_axis(ax, title)


def plot_selected_folder_overview(folder_path, color_mode="pLDDT"):
    model_df = model_tables.get(str(folder_path), pd.DataFrame())
    if model_df.empty:
        raise ValueError(f"No parsed models found for {folder_path}")
    model_df = model_df.sort_values("rank")
    pdb_paths = [Path(p) for p in model_df["pdb_path"]]

    fig = plt.figure(figsize=(12, 4))
    ax1 = fig.add_subplot(1, 2, 1, projection="3d")
    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    plot_chain_colored(ax1, pdb_paths[0], "colored by chain")
    plot_plddt_colored(ax2, pdb_paths[0], "colored by pLDDT")
    plt.tight_layout()
    plt.show()

    n = len(pdb_paths)
    ncols = min(5, max(1, math.ceil(math.sqrt(n))))
    nrows = math.ceil(n / ncols)
    fig = plt.figure(figsize=(3.2 * ncols, 3.0 * nrows))
    for idx, pdb_path in enumerate(pdb_paths, start=1):
        ax = fig.add_subplot(nrows, ncols, idx, projection="3d")
        if color_mode == "Chain A/B":
            plot_chain_colored(ax, pdb_path, f"rank {parse_rank(pdb_path):03d}")
        else:
            plot_plddt_colored(ax, pdb_path, f"rank {parse_rank(pdb_path):03d}")
    mode_title = "chain A/B" if color_mode == "Chain A/B" else "pLDDT"
    plt.suptitle(f"{Path(folder_path).name}: all ranked models colored by {mode_title}", y=0.995, fontsize=14)
    plt.tight_layout()
    plt.show()


plot_output = widgets.Output()
plot_folder_dropdown = widgets.Dropdown(options=folder_options, description="Result:", layout=widgets.Layout(width="650px"))
plot_color_dropdown = widgets.Dropdown(options=["pLDDT", "Chain A/B"], value="pLDDT", description="Color:", layout=widgets.Layout(width="260px"))


def refresh_2d_plots(change=None):
    with plot_output:
        clear_output(wait=True)
        plot_selected_folder_overview(plot_folder_dropdown.value, color_mode=plot_color_dropdown.value)

plot_folder_dropdown.observe(refresh_2d_plots, names="value")
plot_color_dropdown.observe(refresh_2d_plots, names="value")
refresh_2d_plots()
display(widgets.VBox([widgets.HBox([plot_folder_dropdown, plot_color_dropdown]), plot_output]))


## Interactive 3D Viewer

In [7]:
viewer_folder_dropdown = widgets.Dropdown(options=folder_options, description="Result:", layout=widgets.Layout(width="650px"))
viewer_model_dropdown = widgets.Dropdown(description="Model:", layout=widgets.Layout(width="650px"))
viewer_color_dropdown = widgets.Dropdown(options=["pLDDT", "Chain A/B"], value="pLDDT", description="Color:", layout=widgets.Layout(width="260px"))
viewer_hotspots_text = widgets.Text(
    value=",".join(str(h) if not isinstance(h, tuple) else f"{h[0]}{h[1]}" for h in HOTSPOTS),
    description="Hotspots:",
    placeholder="18,200 or A18,A200 or A:18,A:200",
    layout=widgets.Layout(width="650px"),
)
viewer_output = widgets.Output()


def plddt_legend_html():
    return """
    <div style='font-family: sans-serif; line-height: 1.4; margin: 6px 0;'>
      <b>pLDDT colors</b>
      <span style='display:inline-block; margin-left:10px; color:#1f3cff;'>90-100 very high</span>
      <span style='display:inline-block; margin-left:10px; color:#18c6d9;'>70-90 confident</span>
      <span style='display:inline-block; margin-left:10px; color:#b59b00;'>50-70 low</span>
      <span style='display:inline-block; margin-left:10px; color:#ff7d45;'>&lt;50 very low</span>
    </div>
    """


def chain_legend_html():
    target_color = CHAIN_COLORS.get(TARGET_CHAIN, "#888888")
    binder_color = CHAIN_COLORS.get(BINDER_CHAIN, "#888888")
    return f"""
    <div style='font-family: sans-serif; line-height: 1.4; margin: 6px 0;'>
      <b>Chain colors</b>
      <span style='display:inline-block; margin-left:10px; color:{target_color};'>chain {TARGET_CHAIN} target</span>
      <span style='display:inline-block; margin-left:10px; color:{binder_color};'>chain {BINDER_CHAIN} binder</span>
      <span style='display:inline-block; margin-left:10px; color:#888888;'>other chains</span>
    </div>
    """


def hotspot_legend_html(parsed_hotspots):
    if not parsed_hotspots:
        return ""
    labels = ", ".join(f"{chain}{residue}" for chain, residue in parsed_hotspots)
    return f"""
    <div style='font-family: sans-serif; line-height: 1.4; margin: 6px 0;'>
      <b>Hotspots</b>
      <span style='display:inline-block; margin-left:10px; color:#d6a800;'>yellow sticks/cartoon:</span>
      <span style='display:inline-block; margin-left:4px;'>{labels}</span>
    </div>
    """


def parse_hotspot_input(text):
    hotspots = []
    errors = []
    for raw_item in re.split(r"[,;\s]+", str(text).strip()):
        item = raw_item.strip()
        if not item:
            continue
        match = re.fullmatch(r"([A-Za-z])?:?(\d+)", item)
        if not match:
            errors.append(item)
            continue
        chain = match.group(1).upper() if match.group(1) else TARGET_CHAIN
        residue = int(match.group(2))
        hotspots.append((chain, residue))
    return hotspots, errors


def map_viewer_hotspots_to_pdb(pdb_path, reference_pdb, parsed_hotspots):
    if not parsed_hotspots:
        return [], []
    mapped_hotspots, warnings = map_reference_hotspots_to_pdb(pdb_path, reference_pdb, parsed_hotspots)
    mapped_for_viewer = []
    for pdb_chain, pdb_residue, _label, ref_chain, ref_residue, was_mapped in mapped_hotspots:
        if was_mapped and (pdb_chain != ref_chain or pdb_residue != ref_residue):
            label = f"{ref_chain}{ref_residue}->{pdb_chain}{pdb_residue}"
        else:
            label = f"{ref_chain}{ref_residue}"
        mapped_for_viewer.append({
            "pdb_chain": pdb_chain,
            "pdb_residue": pdb_residue,
            "ref_chain": ref_chain,
            "ref_residue": ref_residue,
            "label": label,
            "was_mapped": was_mapped,
        })
    return mapped_for_viewer, warnings


def highlight_hotspots(view, mapped_hotspots):
    for hotspot in mapped_hotspots:
        selection = {"chain": hotspot["pdb_chain"], "resi": hotspot["pdb_residue"]}
        view.addStyle(selection, {"cartoon": {"color": "#ffd21f"}, "stick": {"color": "#ffd21f", "radius": 0.28}})
        view.addLabel(
            hotspot["label"],
            {
                "position": {"chain": hotspot["pdb_chain"], "resi": hotspot["pdb_residue"], "atom": "CA"},
                "fontColor": "black",
                "backgroundColor": "#ffd21f",
                "fontSize": 12,
                "showBackground": True,
            },
        )


def update_viewer_model_options(change=None):
    model_df = model_tables.get(viewer_folder_dropdown.value, pd.DataFrame()).sort_values("rank")
    options = []
    for row in model_df.itertuples():
        label = f"rank {int(row.rank):03d} | ipTM {row.ipTM:.3f} | pAE {row.pAE_interaction:.2f} | pLDDT {row.binder_pLDDT_median:.1f}"
        options.append((label, row.pdb_path))
    viewer_model_dropdown.options = options
    if options:
        viewer_model_dropdown.value = options[0][1]
    render_py3dmol_viewer()


def render_py3dmol_viewer(change=None):
    with viewer_output:
        clear_output(wait=True)
        if not viewer_model_dropdown.value:
            print("No PDB selected.")
            return
        pdb_path = Path(viewer_model_dropdown.value)
        pdb_text = pdb_path.read_text()
        model_df = model_tables.get(viewer_folder_dropdown.value, pd.DataFrame())
        selected = model_df[model_df["pdb_path"] == str(pdb_path)]
        metadata_html = ""
        if not selected.empty:
            row = selected.iloc[0]
            metadata_html = (
                f"<b>{Path(viewer_folder_dropdown.value).name}</b><br>"
                f"rank {int(row['rank']):03d}; ipTM {row['ipTM']:.3f}; "
                f"pAE_interaction {row['pAE_interaction']:.2f}; "
                f"binder median pLDDT {row['binder_pLDDT_median']:.1f}; "
                f"pose RMSD {row['pose_RMSD']:.2f}; "
                f"pocket surface {row['pocket_surface_coverage']:.0%}; "
                f"specificity {row['pocket_specificity']:.0%}; "
                f"A18 lobe diag {row['hotspot18_lobe_coverage']:.0%}; "
                f"A200 lobe diag {row['hotspot200_lobe_coverage']:.0%}"
            )
        parsed_hotspots, hotspot_errors = parse_hotspot_input(viewer_hotspots_text.value)
        summary_row = all_results_df[all_results_df["folder_path"] == viewer_folder_dropdown.value]
        reference_pdb = None if summary_row.empty else summary_row.iloc[0].get("reference_pdb")
        mapped_hotspots, mapping_warnings = map_viewer_hotspots_to_pdb(pdb_path, reference_pdb, parsed_hotspots)
        hotspot_text = ""
        if mapped_hotspots:
            hotspot_text = "<br>highlighted hotspots: " + ", ".join(h["label"] for h in mapped_hotspots)
        error_text = ""
        if hotspot_errors:
            error_text = "<br><span style='color:#b00020;'>Ignored invalid hotspot entries: " + ", ".join(hotspot_errors) + "</span>"
        if mapping_warnings:
            error_text += "<br><span style='color:#b00020;'>" + " ".join(mapping_warnings) + "</span>"
        legend_html = chain_legend_html() if viewer_color_dropdown.value == "Chain A/B" else plddt_legend_html()
        display(HTML(metadata_html + hotspot_text + error_text + legend_html + hotspot_legend_html([(h["pdb_chain"], h["pdb_residue"]) for h in mapped_hotspots])))
        view = py3Dmol.view(width=900, height=620)
        view.addModel(pdb_text, "pdb")
        if viewer_color_dropdown.value == "Chain A/B":
            view.setStyle({}, {"cartoon": {"color": "#888888"}})
            view.setStyle({"chain": TARGET_CHAIN}, {"cartoon": {"color": CHAIN_COLORS.get(TARGET_CHAIN, "#888888")}})
            view.setStyle({"chain": BINDER_CHAIN}, {"cartoon": {"color": CHAIN_COLORS.get(BINDER_CHAIN, "#888888")}})
        else:
            view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})
        highlight_hotspots(view, mapped_hotspots)
        view.zoomTo()
        view.show()

viewer_folder_dropdown.observe(update_viewer_model_options, names="value")
viewer_model_dropdown.observe(render_py3dmol_viewer, names="value")
viewer_color_dropdown.observe(render_py3dmol_viewer, names="value")
viewer_hotspots_text.observe(render_py3dmol_viewer, names="value")
update_viewer_model_options()
display(widgets.VBox([widgets.HBox([viewer_folder_dropdown, viewer_color_dropdown]), viewer_model_dropdown, viewer_hotspots_text, viewer_output]))


## Quick Sanity Checks

In [8]:
# These checks are intentionally lightweight and use the currently available TREX1 example.
example_rows = all_results_df[all_results_df["folder"].eq("TREX1_top_01_9c0cd")]
if not example_rows.empty:
    example = example_rows.iloc[0]
    assert example["binder_id"] == "run_noise0_45", example["binder_id"]
    assert abs(float(example["mpnn_score"]) - 0.9095) < 1e-6, example["mpnn_score"]
    model_df = model_tables[example["folder_path"]].sort_values("rank")
    assert len(model_df) == 20, len(model_df)
    top_model = model_df.iloc[0]
    assert "A18->A15" in top_model["hotspot_mapping"], top_model["hotspot_mapping"]
    assert "A200->A188" in top_model["hotspot_mapping"], top_model["hotspot_mapping"]
    assert int(model_df["model_passed"].sum()) > 0, int(model_df["model_passed"].sum())
    print("Example checks passed: binder_id, mpnn_score, 20 model pairs, mapped hotspots, and pocket-fill pass behavior verified.")
else:
    print("Example folder TREX1_top_01_9c0cd not found; skipped example checks.")


Example checks passed: binder_id, mpnn_score, and 20 model pairs parsed.
